# Dicas

Caso utilize o colab, recomendo a GPU L4 com RAM alta, funciona muito bem! Limitador depende dos modelos utilizados!

# COLAB ONLY!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Instalação de Dependências

In [ ]:
!pip install accelerate bitsandbytes

In [ ]:
!pip install -r "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/requirements.txt" # COLAB ONLY
!pip install -r "./requirements.txt"
#!pip install gliner[stanza] -U # para gliner2

In [ ]:
!pip install --upgrade transformers

In [ ]:
import re
import torch
from huggingface_hub import login

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

import json
import gc
import os
from tqdm import tqdm
from collections import defaultdict
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer

TOKEN_HF = "seu token" #HF token (caso vá usar algum modelo protegido)

MODELO_METRICA = "Qwen/Qwen2.5-7B-Instruct"
MAX_INPUT_TOKENS_ATAQUE = 2048
MAX_NEW_TOKENS_ATAQUE = 1024

# Classes Base e Pré-processamento

In [ ]:
import json
from abc import ABC, abstractmethod

class TextPreprocessor:
    def __init__(self, max_chars=1500, overlap_chars=200):
        self.max_chars = max_chars
        self.overlap_chars = overlap_chars

    def chunk_with_offsets(self, text):
        chunks = []
        start = 0
        text_length = len(text)

        while start < text_length:
            end = min(start + self.max_chars, text_length)

            if end < text_length:
                last_space = text.rfind(' ', start, end)
                if last_space != -1 and last_space > start:
                    end = last_space

            chunk = text[start:end]
            chunks.append({"text": chunk, "start_offset": start})

            if end == text_length:
                break

            start = end - self.overlap_chars

        return chunks

class BaseGeneralWrapper(ABC):
    @abstractmethod
    def extract_entities(self):
        pass

class BaseModelLLMClass(ABC):
    @abstractmethod
    def generate(self, prompt, system_prompt=None, **kwargs):
        pass

    @abstractmethod
    def extract_entities(self, text, labels=None):
        pass

# Wrappers dos Modelos (GLiNER e SpaCy)

In [ ]:
import spacy
import torch
from gliner import GLiNER
#from gliner2 import GLiNER2

spacy.prefer_gpu()

class BaseGlinerWrapper(BaseGeneralWrapper):
    def __init__(self, model_name):
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = GLiNER.from_pretrained(self.model_name).to(self.device)

    def extract_entities(self, text, labels: list, threshold: float):
        entities = self.model.predict_entities(text=text, labels=labels, threshold=threshold)
        return [{"label": ent['label'], "start_offset": ent['start'], "end_offset": ent['end']} for ent in entities]

class Gliner1MultiV21(BaseGeneralWrapper):
    def __init__(self):
        self.model_name = 'urchade/gliner_multi-v2.1'
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = GLiNER.from_pretrained(self.model_name).to(self.device)

    def extract_entities(self, text, labels: list, threshold: float):
        entities = self.model.predict_entities(text=text, labels=labels, threshold=threshold)
        return [{"label": ent['label'], "start_offset": ent['start'], "end_offset": ent['end']} for ent in entities]

class Gliner2MultiV1():
    def __init__(self):
        self.model_name = 'fastino/gliner2-multi-v1'
        self.device = torch.device('cpu')
        self.model = GLiNER2.from_pretrained(self.model_name).to(self.device)

    def extract_entities(self, text, labels: any, threshold: float, include_spans: bool):
        entities_data = self.model.extract_entities(text=text, entity_types=labels, threshold=threshold, include_spans=include_spans)
        result = []
        for label, items in entities_data.get('entities', {}).items():
            for item in items:
                result.append({"span": item['text'], "label": label, "start_offset": item['start'], "end_offset": item['end']})
        return result

class Gliner1NvidiaPII(BaseGeneralWrapper):
    def __init__(self):
        self.model_name = 'nvidia/gliner-PII'
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = GLiNER.from_pretrained(self.model_name).to(self.device)

    def extract_entities(self, text, labels: list, threshold: float):
        entities = self.model.predict_entities(text=text, labels=labels, threshold=threshold)
        return [{"label": ent['label'], "start_offset": ent['start'], "end_offset": ent['end']} for ent in entities]

class Gliner1MultiPII(BaseGeneralWrapper):
    def __init__(self):
        self.model_name = 'urchade/gliner_multi_pii-v1'
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = GLiNER.from_pretrained(self.model_name).to(self.device)

    def extract_entities(self, text, labels: list, threshold: float):
        entities = self.model.predict_entities(text=text, labels=labels, threshold=threshold)
        return [{"label": ent['label'], "start_offset": ent['start'], "end_offset": ent['end']} for ent in entities]

class GlinerXLarge(BaseGeneralWrapper):
    def __init__(self):
        self.model_name = 'knowledgator/gliner-x-large'
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = GLiNER.from_pretrained(self.model_name).to(self.device)

    def extract_entities(self, text, labels: list, threshold: float):
        entities = self.model.predict_entities(text=text, labels=labels, threshold=threshold)
        return [{"label": ent['label'], "start_offset": ent['start'], "end_offset": ent['end']} for ent in entities]

class BaseSpacyWrapper(BaseGeneralWrapper):
    def __init__(self, model_name):
        try:
            self.model_name = model_name
            self.model = spacy.load(self.model_name)
            self.supported_labels = self.model.get_pipe("ner").labels
        except OSError:
            spacy.cli.download(model_name)
            self.model = spacy.load(self.model_name)
            self.supported_labels = self.model.get_pipe("ner").labels

    def extract_entities(self, text, labels : list):
        for label in labels:
            if label not in self.supported_labels:
                raise ValueError(f"Label not supported! label:{label}, supported labels for the model {self.model_name}: {self.supported_labels}")
        doc = self.model(text)
        return [(ent.text, ent.label_, ent.start_char, ent.end_char) for ent in doc.ents]

class SpacyPTNewsSM(BaseGeneralWrapper):
    def __init__(self):
        try:
            self.model_name = 'pt_core_news_sm'
            self.model = spacy.load(self.model_name)
            self.supported_labels = self.model.get_pipe("ner").labels
        except OSError:
            spacy.cli.download(self.model_name)
            self.model = spacy.load(self.model_name)
            self.supported_labels = self.model.get_pipe("ner").labels

    def extract_entities(self, text, labels) -> list[dict[str,str]]:
        for label in labels:
            if label not in self.supported_labels:
                raise ValueError(f"Label not supported! label:{label}, supported labels for the model {self.model_name}: {self.supported_labels}")
        doc = self.model(text)
        return [{"label": ent.label_, "start_offset": ent.start_char, "end_offset": ent.end_char} for ent in doc.ents]

class SpacyPTNewsMD(BaseGeneralWrapper):
    def __init__(self):
        try:
            self.model_name = 'pt_core_news_md'
            self.model = spacy.load(self.model_name)
            self.supported_labels = self.model.get_pipe("ner").labels
        except OSError:
            spacy.cli.download(self.model_name)
            self.model = spacy.load(self.model_name)
            self.supported_labels = self.model.get_pipe("ner").labels

    def extract_entities(self, text, labels) -> list[dict[str,str]]:
        for label in labels:
            if label not in self.supported_labels:
                raise ValueError(f"Label not supported! label:{label}, supported labels for the model {self.model_name}: {self.supported_labels}")
        doc = self.model(text)
        return [{"label": ent.label_, "start_offset": ent.start_char, "end_offset": ent.end_char} for ent in doc.ents]

class SpacyPTNewsLG(BaseGeneralWrapper):
    def __init__(self):
        try:
            self.model_name = 'pt_core_news_lg'
            self.model = spacy.load(self.model_name)
            self.supported_labels = self.model.get_pipe("ner").labels
        except OSError:
            spacy.cli.download(self.model_name)
            self.model = spacy.load(self.model_name)
            self.supported_labels = self.model.get_pipe("ner").labels

    def extract_entities(self, text, labels) -> list[dict[str,str]]:
        for label in labels:
            if label not in self.supported_labels:
                raise ValueError(f"Label not supported! label:{label}, supported labels for the model {self.model_name}: {self.supported_labels}")
        doc = self.model(text)
        return [{"label": ent.label_, "start_offset": ent.start_char, "end_offset": ent.end_char} for ent in doc.ents]

# Wrappers dos Modelos (HuggingFace / LeNER)

In [ ]:
import torch
from transformers import pipeline

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

class BaseHuggingFaceWrapper(BaseGeneralWrapper):
    def __init__(self, model_name, device=None):
        if device is None:
            device = 0 if torch.cuda.is_available() else -1

        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(self.model_name)

        self.nlp = pipeline(
            "ner",
            model=self.model,
            tokenizer=self.tokenizer,
            aggregation_strategy="simple",
            device=device
        )

        self.supported_labels = list(self.nlp.model.config.label2id.keys())
        self.max_length = self.nlp.model.config.max_position_embeddings

    def extract_entities(self, text, labels=None, stride=64):
        if labels:
            for label in labels:
                if label not in self.supported_labels:
                    raise ValueError(f"Label {label} não suportada. Labels do modelo: {self.supported_labels}")

        predictions = self.nlp(
            text,
            stride=stride,
            #max_length=self.max_length
        )

        results = []
        for pred in predictions:
            label = pred['entity_group']
            if labels and label not in labels:
                continue
            results.append({"label": label, "start_offset": pred['start'], "end_offset": pred['end']})

        return results

class bertimbau_finetuned_lener_base(BaseHuggingFaceWrapper):
    def __init__(self, device=None):
        super().__init__("Luciano/bertimbau-base-lener_br", device)

class bertimbau_finetuned_lener_large(BaseHuggingFaceWrapper):
    def __init__(self, device=None):
        super().__init__("Luciano/bertimbau-large-lener_br", device)


class wikineural_multi_ner(BaseGeneralWrapper):
    def __init__(self, model_name="Babelscape/wikineural-multilingual-ner", device=None):
        if device is None:
            device = 0 if torch.cuda.is_available() else -1

        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(self.model_name)

        self.nlp = pipeline(
            "ner",
            model=self.model,
            tokenizer=self.tokenizer,
            grouped_entities=True,
            device=device
        )

        self.supported_labels = list(self.nlp.model.config.label2id.keys())
        self.max_length = self.nlp.model.config.max_position_embeddings

    def extract_entities(self, text, labels=None, stride=64):
        if labels:
            for label in labels:
                if label not in self.supported_labels:
                    raise ValueError(f"Label {label} não suportada. Labels do modelo: {self.supported_labels}")

        predictions = self.nlp(
            text,
            stride=stride,
            #max_length=self.max_length
        )

        results = []
        for pred in predictions:
            label = pred['entity_group']
            if labels and label not in labels:
                continue
            results.append({"label": label, "start_offset": pred['start'], "end_offset": pred['end']})

        return results

# SLM

## ESTRUTURA BASE

In [ ]:
MODELO_NER = "meta-llama/Llama-3.1-8B-Instruct"

MAX_INPUT_TOKENS_NER = 2048
MAX_NEW_TOKENS_NER = 1024

MAX_CHARS_TRECHO_NER = 3000
OVERLAP_CHARS_NER = 500

ROTULOS_PERMITIDOS = {
    "PESSOA",
    "LOCAL",
    "ORGANIZACAO",
    "TEMPO/DATA",
    "VALOR",
    "JURISPRUDENCIA",
    "LEGISLACAO/FUNDAMENTO",
    "PRODUTODELEI",
}


def login_huggingface():
    token = TOKEN_HF

    if token:
        login(token=token)
        print("Login no Hugging Face realizado.")
    else:
        print("HF_TOKEN não encontrado. Se o modelo for restrito, o carregamento pode falhar.")


def limpar_memoria():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def verificar_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA não está disponível. Ative a GPU para rodar este código.")

    nome_gpu = torch.cuda.get_device_name(0)
    memoria_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    print(f"GPU detectada: {nome_gpu}")
    print(f"VRAM total aproximada: {memoria_total:.2f} GB")


def mostrar_memoria_gpu():
    if torch.cuda.is_available():
        alocada = torch.cuda.memory_allocated(0) / (1024 ** 3)
        reservada = torch.cuda.memory_reserved(0) / (1024 ** 3)

        print(f"VRAM alocada: {alocada:.2f} GB")
        print(f"VRAM reservada: {reservada:.2f} GB")


def escolher_compute_dtype():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA não está disponível.")

    if torch.cuda.is_bf16_supported():
        return torch.bfloat16

    return torch.float16


def carregar_modelo_gpu_4bit(nome_modelo):
    """
    Carrega modelo somente na GPU, em 4-bit.
    Não usa CPU offload nem disk offload.
    Se não couber na VRAM, vai dar CUDA out of memory.
    """
    verificar_gpu()

    torch.backends.cuda.matmul.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

    compute_dtype = escolher_compute_dtype()

    tokenizer = AutoTokenizer.from_pretrained(
        nome_modelo,
        trust_remote_code=True
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

    kwargs_modelo = {
        "trust_remote_code": True,
        "device_map": {"": 0},
        "quantization_config": quant_config,
        "low_cpu_mem_usage": True,
    }

    try:
        model = AutoModelForCausalLM.from_pretrained(
            nome_modelo,
            attn_implementation="sdpa",
            **kwargs_modelo
        )
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(
            nome_modelo,
            **kwargs_modelo
        )

    model.eval()
    model.config.use_cache = True

    limpar_memoria()

    print(f"Modelo carregado na GPU: {nome_modelo}")
    mostrar_memoria_gpu()

    return {
        "model": model,
        "tokenizer": tokenizer,
        "nome_modelo": nome_modelo
    }


def aplicar_chat_template(tokenizer, mensagens):
    try:
        return tokenizer.apply_chat_template(
            mensagens,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            mensagens,
            tokenize=False,
            add_generation_prompt=True
        )


def gerar_resposta_chat(
    mensagens,
    gerador,
    max_new_tokens,
    max_input_tokens=2048
):
    model = gerador["model"]
    tokenizer = gerador["tokenizer"]

    prompt_formatado = aplicar_chat_template(tokenizer, mensagens)

    inputs = tokenizer(
        prompt_formatado,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens
    ).to("cuda")

    with torch.inference_mode():
        saida = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    tokens_gerados = saida[0][inputs["input_ids"].shape[-1]:]

    texto_gerado = tokenizer.decode(
        tokens_gerados,
        skip_special_tokens=True
    ).strip()

    del inputs
    del saida
    del tokens_gerados

    return texto_gerado


def montar_prompt(texto):
    return f"""
Você é um sistema de extração de informações altamente preciso.
Sua única tarefa é analisar o texto fornecido e retornar entidades nomeadas em JSON válido.

Rótulos permitidos:
- "PESSOA"
- "LOCAL"
- "ORGANIZACAO"
- "TEMPO/DATA"
- "VALOR"
- "JURISPRUDENCIA"
- "LEGISLACAO/FUNDAMENTO"
- "PRODUTODELEI"

Regras obrigatórias:
1. Retorne apenas entidades encontradas explicitamente no texto.
2. Não modifique, corrija ou normalize o texto extraído.
3. Use apenas os rótulos permitidos.
4. Responda somente com JSON válido.
5. Não use markdown.
6. Não explique o raciocínio.
7. Se nenhuma entidade for encontrada, retorne exatamente:
{{"entities": []}}

Formato obrigatório:
{{"entities": [{{"label": "ROTULO", "text": "texto exato"}}]}}

Exemplo 1:
Entrada:
"aquela autarquia , aplicando o dispositivo a partir de uma leitura literal da norma , indeferia tais requerimentos , o que levou à proposição de diversas ações na justiça federal em que se pleiteava a aplicação extensiva do parágrafo único do art . 34 do estatuto do idoso . 1 art . 29 ."

Saída:
{{"entities": [{{"label": "ORGANIZACAO", "text": "justiça federal"}}, {{"label": "LEGISLACAO/FUNDAMENTO", "text": "art . 34 do estatuto do idoso"}}]}}

Exemplo 2:
Entrada:
"Número do Acórdão ACÓRDÃO 1160/2016 - PLENÁRIO Relator AUGUSTO NARDES Processo 006.010/2000-4 Tipo de processo TOMADA DE CONTAS SIMPLIFICADA ( TCSP ) Data da sessão 11/05/2016 Número da ata 16/2016 Relator da deliberação recorrida Ministra Ana Arraes ."

Saída:
{{"entities": [{{"label": "JURISPRUDENCIA", "text": "ACÓRDÃO 1160/2016"}}, {{"label": "ORGANIZACAO", "text": "PLENÁRIO"}}, {{"label": "PESSOA", "text": "AUGUSTO NARDES"}}, {{"label": "JURISPRUDENCIA", "text": "Processo 006.010/2000-4"}}, {{"label": "TEMPO/DATA", "text": "11/05/2016"}}, {{"label": "PESSOA", "text": "Ana Arraes"}}]}}

Exemplo 3:
Entrada:
"O sistema foi atualizado na versão mais recente sem apresentar falhas de compilação durante a madrugada."

Saída:
{{"entities": []}}

Agora extraia as entidades do texto abaixo.

Entrada:
"{texto}"

Saída:
""".strip()


def gerar_texto_modelo(texto, gerador, max_new_tokens=MAX_NEW_TOKENS_NER):
    prompt = montar_prompt(texto)

    mensagens = [
        {
            "role": "system",
            "content": "Você extrai entidades nomeadas e responde somente JSON válido."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    return gerar_resposta_chat(
        mensagens=mensagens,
        gerador=gerador,
        max_new_tokens=max_new_tokens,
        max_input_tokens=MAX_INPUT_TOKENS_NER
    )


def limpar_saida_modelo(saida):
    saida = str(saida).strip()

    saida = re.sub(
        r"<think>.*?</think>",
        "",
        saida,
        flags=re.DOTALL | re.IGNORECASE
    ).strip()

    saida = saida.replace("```json", "").replace("```", "").strip()

    saida = re.sub(
        r"(?is)thinking process:.*?(?=\{|\[)",
        "",
        saida
    ).strip()

    return saida


def encontrar_json_valido(saida):
    saida = limpar_saida_modelo(saida)

    try:
        return json.loads(saida)
    except json.JSONDecodeError:
        pass

    candidatos = []

    for inicio, caractere_inicial in enumerate(saida):
        if caractere_inicial not in "{[":
            continue

        pilha = []
        dentro_string = False
        escape = False

        for fim in range(inicio, len(saida)):
            c = saida[fim]

            if dentro_string:
                if escape:
                    escape = False
                elif c == "\\":
                    escape = True
                elif c == '"':
                    dentro_string = False
                continue

            if c == '"':
                dentro_string = True
            elif c in "{[":
                pilha.append(c)
            elif c in "}]":
                if not pilha:
                    break

                topo = pilha.pop()

                if topo == "{" and c != "}":
                    break

                if topo == "[" and c != "]":
                    break

                if not pilha:
                    trecho = saida[inicio:fim + 1]

                    try:
                        obj = json.loads(trecho)

                        if isinstance(obj, dict) and "entities" in obj:
                            candidatos.append(obj)
                        elif isinstance(obj, list):
                            candidatos.append(obj)

                    except json.JSONDecodeError:
                        pass

                    break

    if candidatos:
        return candidatos[-1]

    raise ValueError("JSON válido não encontrado na saída do modelo.")


def converter_para_lista_entidades(obj):
    if isinstance(obj, dict):
        entidades = obj.get("entities", [])
    elif isinstance(obj, list):
        entidades = obj
    else:
        entidades = []

    if not isinstance(entidades, list):
        return []

    entidades_limpas = []

    for ent in entidades:
        if not isinstance(ent, dict):
            continue

        label = str(ent.get("label", "")).strip()
        texto_entidade = str(ent.get("text", "")).strip()

        if not label or not texto_entidade:
            continue

        if label not in ROTULOS_PERMITIDOS:
            continue

        entidades_limpas.append({
            "label": label,
            "text": texto_entidade
        })

    return entidades_limpas


def adicionar_offsets(texto, entidades):
    entities = []
    proxima_busca = {}
    vistos = set()

    for ent in entidades:
        label = ent["label"]
        texto_entidade = ent["text"]

        inicio_busca = proxima_busca.get(texto_entidade, 0)
        start = texto.find(texto_entidade, inicio_busca)

        if start == -1:
            start = texto.find(texto_entidade)

        if start == -1:
            continue

        end = start + len(texto_entidade)

        chave = (label, start, end)

        if chave in vistos:
            continue

        vistos.add(chave)
        proxima_busca[texto_entidade] = end

        entities.append({
            "label": label,
            "start": start,
            "end": end
        })

    return entities


def dividir_texto(
    texto,
    max_chars=MAX_CHARS_TRECHO_NER,
    overlap_chars=OVERLAP_CHARS_NER
):
    """
    Divide o texto usando sliding window com sobreposição.

    Exemplo:
    max_chars = 3000
    overlap_chars = 500

    Janela 1: 0    até 3000
    Janela 2: 2500 até 5500
    Janela 3: 5000 até 8000
    """

    if max_chars <= 0:
        raise ValueError("max_chars deve ser maior que zero.")

    if overlap_chars < 0:
        raise ValueError("overlap_chars não pode ser negativo.")

    if overlap_chars >= max_chars:
        raise ValueError("overlap_chars deve ser menor que max_chars.")

    if len(texto) <= max_chars:
        return [(0, texto)]

    partes = []
    stride = max_chars - overlap_chars
    inicio = 0

    while inicio < len(texto):
        fim = min(inicio + max_chars, len(texto))
        trecho = texto[inicio:fim]

        if trecho.strip():
            partes.append((inicio, trecho))

        if fim >= len(texto):
            break

        inicio += stride

    return partes


def extrair_entidades(texto, gerador):
    todas_entities = []
    vistos = set()

    try:
        partes = dividir_texto(
            texto,
            max_chars=MAX_CHARS_TRECHO_NER,
            overlap_chars=OVERLAP_CHARS_NER
        )

        for offset_base, trecho in partes:
            saida_modelo = gerar_texto_modelo(
                trecho,
                gerador,
                max_new_tokens=MAX_NEW_TOKENS_NER
            )

            json_extraido = encontrar_json_valido(saida_modelo)
            entidades = converter_para_lista_entidades(json_extraido)
            entities_trecho = adicionar_offsets(trecho, entidades)

            for ent in entities_trecho:
                start_abs = offset_base + ent["start"]
                end_abs = offset_base + ent["end"]

                chave = (ent["label"], start_abs, end_abs)

                if chave in vistos:
                    continue

                vistos.add(chave)

                todas_entities.append({
                    "label": ent["label"],
                    "start_offset": start_abs,
                    "end_offset": end_abs
                })

        todas_entities = sorted(
            todas_entities,
            key=lambda x: (x["start_offset"], x["end_offset"], x["label"])
        )

        return todas_entities

    except Exception as erro:
        print("Erro na extração:", erro)
        return []

### TESTE

In [ ]:
gerador = carregar_modelo_gpu_4bit(MODELO_NER)

texto = (
        "Acórdão VISTOS , relatados e discutidos estes autos de recurso de reconsideração interposto pelo Sr. Carlos Aureliano Motta de Souza , contra o Acórdão nº 1.466/2013-TCU-Plenário , ACORDAM os Ministros do Tribunal de Contas da União , reunidos em sessão do Plenário , ante as razões expostas pelo Relator , em : 9.1. conhecer , com fundamento nos arts . 32 , inciso I , e 33 da Lei nº 8.443 , de 16 de julho de 1992 , do recurso para , no mérito , dar-lhe provimento , a fim de julgar regulares com ressalva as contas do Senhor Carlos Aureliano Motta de Souza e afastar o débito que lhe foi atribuído mediante o item 9.2 do Acórdão n.º 1.466/2013 – Plenário , tornando insubsistente , outrossim , a multa que lhe foi aplicada , mantendo-se o débito apenas para o Grupo OK Construções e Empreendimentos Ltda. bem como a respectiva multa que lhe foi imputada ; 9.2. dar conhecimento desta deliberação ao recorrente e aos interessados ."
    )

resultado = extrair_entidades(texto, gerador)

print(json.dumps(resultado, indent=4, ensure_ascii=False))

# NER V3

In [ ]:
class PrivacyMetricsEvaluator:
    def __init__(self, sbert_device="cuda"):

        self.sbert_model = SentenceTransformer(
            "paraphrase-multilingual-MiniLM-L12-v2",
            device=sbert_device
        )
        self.model_metrica  = carregar_modelo_gpu_4bit(MODELO_METRICA)
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

    def generate_anonymized_text(self, text: str, entities: list) -> str:
        sorted_entities = sorted(
            entities,
            key=lambda x: x["start_offset"],
            reverse=True
        )

        anonymized_text = text

        for ent in sorted_entities:
            start = ent["start_offset"]
            end = ent["end_offset"]

            anonymized_text = (
                anonymized_text[:start]
                + f"[{ent['label']}]"
                + anonymized_text[end:]
            )

        return anonymized_text

    def calculate_sbert_tps(self, original_text: str, anonymized_text: str) -> float:
        with torch.inference_mode():
            emb1 = self.sbert_model.encode(
                original_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            emb2 = self.sbert_model.encode(
                anonymized_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            cosine_scores = util.cos_sim(emb1, emb2)
            resultado = cosine_scores.item()

        del emb1
        del emb2
        del cosine_scores

        return resultado

    def calculate_lexical_divergence(self, original_text: str, anonymized_text: str) -> float:
        set_orig = set(original_text.lower().split())
        set_anon = set(anonymized_text.lower().split())

        intersection = len(set_orig.intersection(set_anon))
        union = len(set_orig.union(set_anon))

        if union == 0:
            return 0.0

        jaccard_similarity = intersection / union

        return 1.0 - jaccard_similarity

    def calculate_rouge_overlap(self, original_text: str, anonymized_text: str) -> dict:
        scores = self.rouge.score(original_text, anonymized_text)

        return {
            "rouge1_fmeasure": scores["rouge1"].fmeasure,
            "rougeL_fmeasure": scores["rougeL"].fmeasure
        }

    def execute_llm_reidentification_attack(
        self,
        original_text: str,
        anonymized_text: str,
        true_entities: list,
        use_original: bool
    ) -> float:
        if not true_entities:
            return 0.0

        original_text_prompt = original_text[:2500]
        anonymized_text_prompt = anonymized_text[:2500]

        if use_original:
            prompt = f"""
Você é um especialista em reidentificação.
Mapeie as tags do texto anonimizado para as palavras reais usando o texto original.

Texto Original:
{original_text_prompt}

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com os valores reais removidos, separados por vírgula.
""".strip()
        else:
            prompt = f"""
Você é um especialista em reidentificação.
Tente adivinhar as palavras originais que foram substituídas por tags no seguinte texto.

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com seus palpites, separados por vírgula.
""".strip()

        mensagens = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        try:
            generated_text = gerar_resposta_chat(
                mensagens=mensagens,
                gerador=self.model_metrica,
                max_new_tokens=MAX_NEW_TOKENS_ATAQUE,
                max_input_tokens=MAX_INPUT_TOKENS_ATAQUE
            ).lower()

        except Exception:
            return 0.0

        success_count = 0

        for ent in true_entities:
            entity_text = original_text[
                ent["start_offset"]:ent["end_offset"]
            ].lower()

            if entity_text in generated_text:
                success_count += 1

        return success_count / len(true_entities)

    def avaliar_texto(self, original_text: str, entities: list) -> dict:
        anonymized_text = self.generate_anonymized_text(
            original_text,
            entities
        )

        sbert_tps = self.calculate_sbert_tps(
            original_text,
            anonymized_text
        )

        lexical_divergence = self.calculate_lexical_divergence(
            original_text,
            anonymized_text
        )

        rouge_scores = self.calculate_rouge_overlap(
            original_text,
            anonymized_text
        )

        ataque_com_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=True
        )

        ataque_sem_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=False
        )

        return {
            "original_text": original_text,
            "anonymized_text": anonymized_text,
            "entities": entities,
            "sbert_tps": sbert_tps,
            "lexical_divergence": lexical_divergence,
            "rouge1_fmeasure": rouge_scores["rouge1_fmeasure"],
            "rougeL_fmeasure": rouge_scores["rougeL_fmeasure"],
            "llm_attack_with_original": ataque_com_original,
            "llm_attack_without_original": ataque_sem_original
        }


# Implemente sua lógica aqui
# Use os exemplos abaixo para fornecer a mesma saida da função predict!
class NERModel:
    def __init__(self):
        self.save_arq = ''
        self.model_used =


    def predict(self, text: str):
        continue



def calculate_metrics(true_entities, pred_entities):
    true_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in true_entities)
    pred_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in pred_entities)

    tp_set = true_set.intersection(pred_set)
    fp_set = pred_set - true_set
    fn_set = true_set - pred_set

    label_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for label, _, _ in tp_set:
        label_metrics[label]['tp'] += 1

    for label, _, _ in fp_set:
        label_metrics[label]['fp'] += 1

    for label, _, _ in fn_set:
        label_metrics[label]['fn'] += 1

    return label_metrics

def evaluate_model(dataset_path, dataset_name, model, batch_size=10):
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    privacy_evaluator = PrivacyMetricsEvaluator()

    global_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    privacy_metrics = {
        "sbert_tps_sum": 0.0,
        "lexical_divergence_sum": 0.0,
        "rouge1_sum": 0.0,
        "rougeL_sum": 0.0,
        "llm_reid_no_orig_sum": 0.0,
        "llm_reid_with_orig_sum": 0.0,
        "processed_docs": 0
    }

    attack_dataset = []
    log_dataset = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Processando {dataset_name}"):
        batch = dataset[i:i + batch_size]

        for doc in batch:
            text = doc.get('doc_text', '')
            true_entities = doc.get('entities', [])
            pred_entities = model.predict(text)

            doc_metrics = calculate_metrics(true_entities, pred_entities)
            for label, counts in doc_metrics.items():
                global_metrics[label]['tp'] += counts['tp']
                global_metrics[label]['fp'] += counts['fp']
                global_metrics[label]['fn'] += counts['fn']

            anonymized_text = privacy_evaluator.generate_anonymized_text(text, pred_entities)

            tps_score = privacy_evaluator.calculate_sbert_tps(text, anonymized_text)
            div_score = privacy_evaluator.calculate_lexical_divergence(text, anonymized_text)
            rouge_scores = privacy_evaluator.calculate_rouge_overlap(text, anonymized_text)

            llm_rate_no_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=False)
            llm_rate_with_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=True)

            privacy_metrics["sbert_tps_sum"] += tps_score
            privacy_metrics["lexical_divergence_sum"] += div_score
            privacy_metrics["rouge1_sum"] += rouge_scores["rouge1_fmeasure"]
            privacy_metrics["rougeL_sum"] += rouge_scores["rougeL_fmeasure"]
            privacy_metrics["llm_reid_no_orig_sum"] += llm_rate_no_orig
            privacy_metrics["llm_reid_with_orig_sum"] += llm_rate_with_orig
            privacy_metrics["processed_docs"] += 1

            attack_dataset.append({
                "original_text": text,
                "anonymized_text": anonymized_text,
                "true_entities": true_entities
            })

            log_dataset.append({
                "original_text": text,
                "true_entities": true_entities,
                "predicted_entities": pred_entities
            })

        gc.collect()

    total_tp = total_fp = total_fn = 0
    results_to_save = {"per_entity": {}, "global": {}, "privacy_and_utility": {}}

    print("=" * 40)
    print(f"RESULTADOS POR ENTIDADE - {dataset_name}")
    print("=" * 40)

    for label, counts in global_metrics.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
        total_tp += tp
        total_fp += fp
        total_fn += fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        print(f"Entidade: [{label}] | F1: {f1:.4f}")

        results_to_save["per_entity"][label] = {
            "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1_score": round(f1, 4)
        }

    g_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    g_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    g_f1 = 2 * (g_precision * g_recall) / (g_precision + g_recall) if (g_precision + g_recall) > 0 else 0.0

    docs = privacy_metrics["processed_docs"]
    avg_tps = privacy_metrics["sbert_tps_sum"] / docs if docs > 0 else 0
    avg_div = privacy_metrics["lexical_divergence_sum"] / docs if docs > 0 else 0
    avg_rouge1 = privacy_metrics["rouge1_sum"] / docs if docs > 0 else 0
    avg_rougeL = privacy_metrics["rougeL_sum"] / docs if docs > 0 else 0
    avg_llm_no_orig = privacy_metrics["llm_reid_no_orig_sum"] / docs if docs > 0 else 0
    avg_llm_with_orig = privacy_metrics["llm_reid_with_orig_sum"] / docs if docs > 0 else 0

    results_to_save["global"] = {
        "total_tp": total_tp, "total_fp": total_fp, "total_fn": total_fn,
        "precision": round(g_precision, 4),
        "recall": round(g_recall, 4),
        "f1_score": round(g_f1, 4)
    }
    try:
        results_to_save["privacy_and_utility"] = {
            "sbert_tps_avg": round(avg_tps, 4),
            "lexical_divergence_avg": round(avg_div, 4),
            "rouge1_overlap_avg": round(avg_rouge1, 4),
            "rougeL_overlap_avg": round(avg_rougeL, 4),
            f"{MODELO_METRICA}_reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
            f"{MODELO_METRICA}_reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    except:
        results_to_save["privacy_and_utility"] = {
          "sbert_tps_avg": round(avg_tps, 4),
          "lexical_divergence_avg": round(avg_div, 4),
          "rouge1_overlap_avg": round(avg_rouge1, 4),
          "rougeL_overlap_avg": round(avg_rougeL, 4),
          "reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
          "reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    save_dir = f'{dataset_name}/' # forneça o local de salvamento!
    os.makedirs(save_dir, exist_ok=True)
    saved_in = f'{save_dir}geral_info_{model.save_arq}.json'
    attack_dataset_path = f'{save_dir}attack_dataset_{model.save_arq}.json'
    log_dataset_path = f'{save_dir}log_info_{model.save_arq}.json'

    with open(saved_in, 'w', encoding='utf-8') as f_out:
        json.dump(results_to_save, f_out, indent=4, ensure_ascii=False)

    with open(attack_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(attack_dataset, f_out, indent=4, ensure_ascii=False)

    with open(log_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(log_dataset, f_out, indent=4, ensure_ascii=False)
    try:
        print("=" * 40)
        print(f"F1 Global ({dataset_name}): {g_f1:.4f}")
        print(f"TPS (SBERT): {avg_tps:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Sem Orig): {avg_llm_no_orig:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Com Orig): {avg_llm_with_orig:.4f}")
        print(f"Arquivo de resultados gerado com sucesso: {saved_in}")
        print(f"Arquivo de ataque gerado com sucesso: {attack_dataset_path}")
        print(f"Arquivo de log gerado com sucesso: {log_dataset_path}\n")
    except:
      pass

datasets = [
    #{"name": "first_harem", "path": "./datasets/json_format_v2/first_harem/first_harem_selective.json"},
    #{"name": "lener_br", "path": "./datasets/json_format_v2/lener_br/lener_br_converted.json"},
    #{"name": "mariNER", "path": "./datasets/json_format_v2/mariNER/mariNER_converted.json"},
    #{"name": "mini_harem", "path": "./datasets/json_format_v2/mini_harem/mini_harem_selective.json"},
    #{"name": "second_harem_selective", "path": "./datasets/json_format_v2/second_harem/second_harem_selective.json"},
    #{"name": "ulyssesNER_Br_PL_v2_corpus", "path": "./datasets/json_format_v2/ulyssesNER_Br_PL_v2_corpus/ulyssesCategorias-v2_converted.json"},
]

meu_modelo = NERModel()

for ds in datasets:
    print(f"Iniciando avaliação: {ds['name']}")
    evaluate_model(dataset_path=ds['path'], dataset_name=ds['name'], model=meu_modelo, batch_size=5)

# Exemplos de Execução

## GLINER2

In [ ]:
class PrivacyMetricsEvaluator:
    def __init__(self, sbert_device="cuda"):

        self.sbert_model = SentenceTransformer(
            "paraphrase-multilingual-MiniLM-L12-v2",
            device=sbert_device
        )
        self.model_metrica  = carregar_modelo_gpu_4bit(MODELO_METRICA)
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

    def generate_anonymized_text(self, text: str, entities: list) -> str:
        sorted_entities = sorted(
            entities,
            key=lambda x: x["start_offset"],
            reverse=True
        )

        anonymized_text = text

        for ent in sorted_entities:
            start = ent["start_offset"]
            end = ent["end_offset"]

            anonymized_text = (
                anonymized_text[:start]
                + f"[{ent['label']}]"
                + anonymized_text[end:]
            )

        return anonymized_text

    def calculate_sbert_tps(self, original_text: str, anonymized_text: str) -> float:
        with torch.inference_mode():
            emb1 = self.sbert_model.encode(
                original_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            emb2 = self.sbert_model.encode(
                anonymized_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            cosine_scores = util.cos_sim(emb1, emb2)
            resultado = cosine_scores.item()

        del emb1
        del emb2
        del cosine_scores

        return resultado

    def calculate_lexical_divergence(self, original_text: str, anonymized_text: str) -> float:
        set_orig = set(original_text.lower().split())
        set_anon = set(anonymized_text.lower().split())

        intersection = len(set_orig.intersection(set_anon))
        union = len(set_orig.union(set_anon))

        if union == 0:
            return 0.0

        jaccard_similarity = intersection / union

        return 1.0 - jaccard_similarity

    def calculate_rouge_overlap(self, original_text: str, anonymized_text: str) -> dict:
        scores = self.rouge.score(original_text, anonymized_text)

        return {
            "rouge1_fmeasure": scores["rouge1"].fmeasure,
            "rougeL_fmeasure": scores["rougeL"].fmeasure
        }

    def execute_llm_reidentification_attack(
        self,
        original_text: str,
        anonymized_text: str,
        true_entities: list,
        use_original: bool
    ) -> float:
        if not true_entities:
            return 0.0

        original_text_prompt = original_text[:2500]
        anonymized_text_prompt = anonymized_text[:2500]

        if use_original:
            prompt = f"""
Você é um especialista em reidentificação.
Mapeie as tags do texto anonimizado para as palavras reais usando o texto original.

Texto Original:
{original_text_prompt}

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com os valores reais removidos, separados por vírgula.
""".strip()
        else:
            prompt = f"""
Você é um especialista em reidentificação.
Tente adivinhar as palavras originais que foram substituídas por tags no seguinte texto.

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com seus palpites, separados por vírgula.
""".strip()

        mensagens = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        try:
            generated_text = gerar_resposta_chat(
                mensagens=mensagens,
                gerador=self.model_metrica,
                max_new_tokens=MAX_NEW_TOKENS_ATAQUE,
                max_input_tokens=MAX_INPUT_TOKENS_ATAQUE
            ).lower()

        except Exception:
            return 0.0

        success_count = 0

        for ent in true_entities:
            entity_text = original_text[
                ent["start_offset"]:ent["end_offset"]
            ].lower()

            if entity_text in generated_text:
                success_count += 1

        return success_count / len(true_entities)

    def avaliar_texto(self, original_text: str, entities: list) -> dict:
        anonymized_text = self.generate_anonymized_text(
            original_text,
            entities
        )

        sbert_tps = self.calculate_sbert_tps(
            original_text,
            anonymized_text
        )

        lexical_divergence = self.calculate_lexical_divergence(
            original_text,
            anonymized_text
        )

        rouge_scores = self.calculate_rouge_overlap(
            original_text,
            anonymized_text
        )

        ataque_com_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=True
        )

        ataque_sem_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=False
        )

        return {
            "original_text": original_text,
            "anonymized_text": anonymized_text,
            "entities": entities,
            "sbert_tps": sbert_tps,
            "lexical_divergence": lexical_divergence,
            "rouge1_fmeasure": rouge_scores["rouge1_fmeasure"],
            "rougeL_fmeasure": rouge_scores["rougeL_fmeasure"],
            "llm_attack_with_original": ataque_com_original,
            "llm_attack_without_original": ataque_sem_original
        }

class NERModel:
    def __init__(self):
        self.save_arq = 'gliner2_multi-v1_v1'
        self.gliner2_multiv1 = Gliner2MultiV1()
        self.threshold = 0.3
        self.include_spans = True

        self.schema = {
            "person": {
                "description": "Specific, proper names of people. First names, last names, nicknames, and sometimes titles or honorifics attached to the name.",
            },
            "organization": {
                "description": "Companies, institutions, government bodies, sports teams, and legal entities. Full company names, acronyms, or commonly known abbreviations of institutions.",
            },
            "local": {
                "description": "Countries, cities, states, provinces, and counties. Physical, natural, or geographical features that are not technically political entities.",
            },
            "time_or_data": {
                "description": "Any temporal reference, including specific civil calendar dates (days, months, years), times, eras, or historical periods."
            },
            "value" :{
                "description": "Expressions denoting a numerical value or quantity. Includes ages (e.g., '78 years'), measurements and distances (e.g., '2.5 km'), time in sports or geographical notation (e.g., '14''), ordinals, and cardinals (e.g., '20', '26th')."
            },
            "case_law": {
                "description": "Any reference to decisions of courts and judging bodies, including appellate decisions (acórdãos), binding precedents (súmulas), judicial sentences, and procedural control numbers."
            },
            "legislation_or_legal_basis" : {
                "description": "Any reference to legal, normative, or constitutional documents, including laws, decrees, internal regulations, codes, articles, paragraphs, and clauses that serve as a legal basis."
            },
            "product_of_law": {
                "description": "Any reference to government programs, public funds, demarcated zones, titles, regimes, or institutional mechanisms created and established by law."
            }
        }

    def predict(self, text: str):
        pred = self.gliner2_multiv1.extract_entities(text=text, labels=self.schema, threshold=self.threshold, include_spans=self.include_spans)

        for p in pred:
            if p['label'] == "person":
                p['label'] = "PESSOA"
            elif p['label'] == "organization":
                p['label'] = "ORGANIZACAO"
            elif p['label'] == "local":
                p['label'] = "LOCAL"
            elif p['label'] == 'time_or_data':
              p['label'] = "TEMPO/DATA"
            elif p['label'] == 'value':
              p['label'] = "VALOR"
            elif p['label'] == 'case_law':
              p['label'] = "JURISPRUDENCIA"
            elif p['label'] == 'legislation_or_legal_basis':
              p['label'] = "LEGISLACAO/FUNDAMENTO"
            elif p['label'] == 'product_of_law':
              p['label'] = "PRODUTODELEI"

        return pred

def calculate_metrics(true_entities, pred_entities):
    true_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in true_entities)
    pred_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in pred_entities)

    tp_set = true_set.intersection(pred_set)
    fp_set = pred_set - true_set
    fn_set = true_set - pred_set

    label_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for label, _, _ in tp_set:
        label_metrics[label]['tp'] += 1

    for label, _, _ in fp_set:
        label_metrics[label]['fp'] += 1

    for label, _, _ in fn_set:
        label_metrics[label]['fn'] += 1

    return label_metrics

def evaluate_model(dataset_path, dataset_name, model, batch_size=10):
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    privacy_evaluator = PrivacyMetricsEvaluator()

    global_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    privacy_metrics = {
        "sbert_tps_sum": 0.0,
        "lexical_divergence_sum": 0.0,
        "rouge1_sum": 0.0,
        "rougeL_sum": 0.0,
        "llm_reid_no_orig_sum": 0.0,
        "llm_reid_with_orig_sum": 0.0,
        "processed_docs": 0
    }

    attack_dataset = []
    log_dataset = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Processando {dataset_name}"):
        batch = dataset[i:i + batch_size]

        for doc in batch:
            text = doc.get('doc_text', '')
            true_entities = doc.get('entities', [])
            pred_entities = model.predict(text)

            doc_metrics = calculate_metrics(true_entities, pred_entities)
            for label, counts in doc_metrics.items():
                global_metrics[label]['tp'] += counts['tp']
                global_metrics[label]['fp'] += counts['fp']
                global_metrics[label]['fn'] += counts['fn']

            anonymized_text = privacy_evaluator.generate_anonymized_text(text, pred_entities)

            tps_score = privacy_evaluator.calculate_sbert_tps(text, anonymized_text)
            div_score = privacy_evaluator.calculate_lexical_divergence(text, anonymized_text)
            rouge_scores = privacy_evaluator.calculate_rouge_overlap(text, anonymized_text)

            llm_rate_no_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=False)
            llm_rate_with_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=True)

            privacy_metrics["sbert_tps_sum"] += tps_score
            privacy_metrics["lexical_divergence_sum"] += div_score
            privacy_metrics["rouge1_sum"] += rouge_scores["rouge1_fmeasure"]
            privacy_metrics["rougeL_sum"] += rouge_scores["rougeL_fmeasure"]
            privacy_metrics["llm_reid_no_orig_sum"] += llm_rate_no_orig
            privacy_metrics["llm_reid_with_orig_sum"] += llm_rate_with_orig
            privacy_metrics["processed_docs"] += 1

            attack_dataset.append({
                "original_text": text,
                "anonymized_text": anonymized_text,
                "true_entities": true_entities
            })

            log_dataset.append({
                "original_text": text,
                "true_entities": true_entities,
                "predicted_entities": pred_entities
            })

        gc.collect()

    total_tp = total_fp = total_fn = 0
    results_to_save = {"per_entity": {}, "global": {}, "privacy_and_utility": {}}

    print("=" * 40)
    print(f"RESULTADOS POR ENTIDADE - {dataset_name}")
    print("=" * 40)

    for label, counts in global_metrics.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
        total_tp += tp
        total_fp += fp
        total_fn += fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        print(f"Entidade: [{label}] | F1: {f1:.4f}")

        results_to_save["per_entity"][label] = {
            "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1_score": round(f1, 4)
        }

    g_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    g_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    g_f1 = 2 * (g_precision * g_recall) / (g_precision + g_recall) if (g_precision + g_recall) > 0 else 0.0

    docs = privacy_metrics["processed_docs"]
    avg_tps = privacy_metrics["sbert_tps_sum"] / docs if docs > 0 else 0
    avg_div = privacy_metrics["lexical_divergence_sum"] / docs if docs > 0 else 0
    avg_rouge1 = privacy_metrics["rouge1_sum"] / docs if docs > 0 else 0
    avg_rougeL = privacy_metrics["rougeL_sum"] / docs if docs > 0 else 0
    avg_llm_no_orig = privacy_metrics["llm_reid_no_orig_sum"] / docs if docs > 0 else 0
    avg_llm_with_orig = privacy_metrics["llm_reid_with_orig_sum"] / docs if docs > 0 else 0

    results_to_save["global"] = {
        "total_tp": total_tp, "total_fp": total_fp, "total_fn": total_fn,
        "precision": round(g_precision, 4),
        "recall": round(g_recall, 4),
        "f1_score": round(g_f1, 4)
    }
    try:
        results_to_save["privacy_and_utility"] = {
            "sbert_tps_avg": round(avg_tps, 4),
            "lexical_divergence_avg": round(avg_div, 4),
            "rouge1_overlap_avg": round(avg_rouge1, 4),
            "rougeL_overlap_avg": round(avg_rougeL, 4),
            f"{MODELO_METRICA}_reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
            f"{MODELO_METRICA}_reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    except:
        results_to_save["privacy_and_utility"] = {
          "sbert_tps_avg": round(avg_tps, 4),
          "lexical_divergence_avg": round(avg_div, 4),
          "rouge1_overlap_avg": round(avg_rouge1, 4),
          "rougeL_overlap_avg": round(avg_rougeL, 4),
          "reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
          "reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    save_dir = f'/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/resultados_v5/{dataset_name}/'
    os.makedirs(save_dir, exist_ok=True)
    saved_in = f'{save_dir}geral_info_{model.save_arq}.json'
    attack_dataset_path = f'{save_dir}attack_dataset_{model.save_arq}.json'
    log_dataset_path = f'{save_dir}log_info_{model.save_arq}.json'

    with open(saved_in, 'w', encoding='utf-8') as f_out:
        json.dump(results_to_save, f_out, indent=4, ensure_ascii=False)

    with open(attack_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(attack_dataset, f_out, indent=4, ensure_ascii=False)

    with open(log_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(log_dataset, f_out, indent=4, ensure_ascii=False)
    try:
        print("=" * 40)
        print(f"F1 Global ({dataset_name}): {g_f1:.4f}")
        print(f"TPS (SBERT): {avg_tps:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Sem Orig): {avg_llm_no_orig:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Com Orig): {avg_llm_with_orig:.4f}")
        print(f"Arquivo de resultados gerado com sucesso: {saved_in}")
        print(f"Arquivo de ataque gerado com sucesso: {attack_dataset_path}")
        print(f"Arquivo de log gerado com sucesso: {log_dataset_path}\n")
    except:
      pass

datasets = [
    #{"name": "first_harem", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/first_harem/first_harem_selective.json"},
    #{"name": "lener_br", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/lener_br/lener_br_converted.json"},
    #{"name": "mariNER", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/mariNER/mariNER_converted.json"},
    #{"name": "mini_harem", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/mini_harem/mini_harem_selective.json"},
    #{"name": "second_harem_selective", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/second_harem/second_harem_selective.json"},
    #{"name": "ulyssesNER_Br_PL_v2_corpus", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/ulyssesNER_Br_PL_v2_corpus/ulyssesCategorias-v2_converted.json"},
]

meu_modelo = NERModel()

for ds in datasets:
    print(f"Iniciando avaliação: {ds['name']}")
    evaluate_model(dataset_path=ds['path'], dataset_name=ds['name'], model=meu_modelo, batch_size=5)

In [ ]:
class PrivacyMetricsEvaluator:
    def __init__(self, sbert_device="cuda"):

        self.sbert_model = SentenceTransformer(
            "paraphrase-multilingual-MiniLM-L12-v2",
            device=sbert_device
        )
        self.model_metrica  = carregar_modelo_gpu_4bit(MODELO_METRICA)
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

    def generate_anonymized_text(self, text: str, entities: list) -> str:
        sorted_entities = sorted(
            entities,
            key=lambda x: x["start_offset"],
            reverse=True
        )

        anonymized_text = text

        for ent in sorted_entities:
            start = ent["start_offset"]
            end = ent["end_offset"]

            anonymized_text = (
                anonymized_text[:start]
                + f"[{ent['label']}]"
                + anonymized_text[end:]
            )

        return anonymized_text

    def calculate_sbert_tps(self, original_text: str, anonymized_text: str) -> float:
        with torch.inference_mode():
            emb1 = self.sbert_model.encode(
                original_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            emb2 = self.sbert_model.encode(
                anonymized_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            cosine_scores = util.cos_sim(emb1, emb2)
            resultado = cosine_scores.item()

        del emb1
        del emb2
        del cosine_scores

        return resultado

    def calculate_lexical_divergence(self, original_text: str, anonymized_text: str) -> float:
        set_orig = set(original_text.lower().split())
        set_anon = set(anonymized_text.lower().split())

        intersection = len(set_orig.intersection(set_anon))
        union = len(set_orig.union(set_anon))

        if union == 0:
            return 0.0

        jaccard_similarity = intersection / union

        return 1.0 - jaccard_similarity

    def calculate_rouge_overlap(self, original_text: str, anonymized_text: str) -> dict:
        scores = self.rouge.score(original_text, anonymized_text)

        return {
            "rouge1_fmeasure": scores["rouge1"].fmeasure,
            "rougeL_fmeasure": scores["rougeL"].fmeasure
        }

    def execute_llm_reidentification_attack(
        self,
        original_text: str,
        anonymized_text: str,
        true_entities: list,
        use_original: bool
    ) -> float:
        if not true_entities:
            return 0.0

        original_text_prompt = original_text[:2500]
        anonymized_text_prompt = anonymized_text[:2500]

        if use_original:
            prompt = f"""
Você é um especialista em reidentificação.
Mapeie as tags do texto anonimizado para as palavras reais usando o texto original.

Texto Original:
{original_text_prompt}

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com os valores reais removidos, separados por vírgula.
""".strip()
        else:
            prompt = f"""
Você é um especialista em reidentificação.
Tente adivinhar as palavras originais que foram substituídas por tags no seguinte texto.

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com seus palpites, separados por vírgula.
""".strip()

        mensagens = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        try:
            generated_text = gerar_resposta_chat(
                mensagens=mensagens,
                gerador=self.model_metrica,
                max_new_tokens=MAX_NEW_TOKENS_ATAQUE,
                max_input_tokens=MAX_INPUT_TOKENS_ATAQUE
            ).lower()

        except Exception:
            return 0.0

        success_count = 0

        for ent in true_entities:
            entity_text = original_text[
                ent["start_offset"]:ent["end_offset"]
            ].lower()

            if entity_text in generated_text:
                success_count += 1

        return success_count / len(true_entities)

    def avaliar_texto(self, original_text: str, entities: list) -> dict:
        anonymized_text = self.generate_anonymized_text(
            original_text,
            entities
        )

        sbert_tps = self.calculate_sbert_tps(
            original_text,
            anonymized_text
        )

        lexical_divergence = self.calculate_lexical_divergence(
            original_text,
            anonymized_text
        )

        rouge_scores = self.calculate_rouge_overlap(
            original_text,
            anonymized_text
        )

        ataque_com_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=True
        )

        ataque_sem_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=False
        )

        return {
            "original_text": original_text,
            "anonymized_text": anonymized_text,
            "entities": entities,
            "sbert_tps": sbert_tps,
            "lexical_divergence": lexical_divergence,
            "rouge1_fmeasure": rouge_scores["rouge1_fmeasure"],
            "rougeL_fmeasure": rouge_scores["rougeL_fmeasure"],
            "llm_attack_with_original": ataque_com_original,
            "llm_attack_without_original": ataque_sem_original
        }

class NERModel:
    def __init__(self):
        self.save_arq = 'pt_core_news_spacy_lg'
        self.model_used = SpacyPTNewsLG()
        self.labels = ['PER', 'ORG', 'LOC']


    def predict(self, text: str):
        pred = self.model_used.extract_entities(text=text, labels=self.labels)

        for p in pred:
            if p['label'] == "PER":
                p['label'] = "PESSOA"
            elif p['label'] == "ORG":
                p['label'] = "ORGANIZACAO"
            elif p['label'] == "LOC":
                p['label'] = "LOCAL"

        return pred

def calculate_metrics(true_entities, pred_entities):
    true_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in true_entities)
    pred_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in pred_entities)

    tp_set = true_set.intersection(pred_set)
    fp_set = pred_set - true_set
    fn_set = true_set - pred_set

    label_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for label, _, _ in tp_set:
        label_metrics[label]['tp'] += 1

    for label, _, _ in fp_set:
        label_metrics[label]['fp'] += 1

    for label, _, _ in fn_set:
        label_metrics[label]['fn'] += 1

    return label_metrics

def evaluate_model(dataset_path, dataset_name, model, batch_size=10):
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    privacy_evaluator = PrivacyMetricsEvaluator()

    global_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    privacy_metrics = {
        "sbert_tps_sum": 0.0,
        "lexical_divergence_sum": 0.0,
        "rouge1_sum": 0.0,
        "rougeL_sum": 0.0,
        "llm_reid_no_orig_sum": 0.0,
        "llm_reid_with_orig_sum": 0.0,
        "processed_docs": 0
    }

    attack_dataset = []
    log_dataset = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Processando {dataset_name}"):
        batch = dataset[i:i + batch_size]

        for doc in batch:
            text = doc.get('doc_text', '')
            true_entities = doc.get('entities', [])
            pred_entities = model.predict(text)

            doc_metrics = calculate_metrics(true_entities, pred_entities)
            for label, counts in doc_metrics.items():
                global_metrics[label]['tp'] += counts['tp']
                global_metrics[label]['fp'] += counts['fp']
                global_metrics[label]['fn'] += counts['fn']

            anonymized_text = privacy_evaluator.generate_anonymized_text(text, pred_entities)

            tps_score = privacy_evaluator.calculate_sbert_tps(text, anonymized_text)
            div_score = privacy_evaluator.calculate_lexical_divergence(text, anonymized_text)
            rouge_scores = privacy_evaluator.calculate_rouge_overlap(text, anonymized_text)

            llm_rate_no_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=False)
            llm_rate_with_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=True)

            privacy_metrics["sbert_tps_sum"] += tps_score
            privacy_metrics["lexical_divergence_sum"] += div_score
            privacy_metrics["rouge1_sum"] += rouge_scores["rouge1_fmeasure"]
            privacy_metrics["rougeL_sum"] += rouge_scores["rougeL_fmeasure"]
            privacy_metrics["llm_reid_no_orig_sum"] += llm_rate_no_orig
            privacy_metrics["llm_reid_with_orig_sum"] += llm_rate_with_orig
            privacy_metrics["processed_docs"] += 1

            attack_dataset.append({
                "original_text": text,
                "anonymized_text": anonymized_text,
                "true_entities": true_entities
            })

            log_dataset.append({
                "original_text": text,
                "true_entities": true_entities,
                "predicted_entities": pred_entities
            })

        gc.collect()

    total_tp = total_fp = total_fn = 0
    results_to_save = {"per_entity": {}, "global": {}, "privacy_and_utility": {}}

    print("=" * 40)
    print(f"RESULTADOS POR ENTIDADE - {dataset_name}")
    print("=" * 40)

    for label, counts in global_metrics.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
        total_tp += tp
        total_fp += fp
        total_fn += fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        print(f"Entidade: [{label}] | F1: {f1:.4f}")

        results_to_save["per_entity"][label] = {
            "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1_score": round(f1, 4)
        }

    g_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    g_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    g_f1 = 2 * (g_precision * g_recall) / (g_precision + g_recall) if (g_precision + g_recall) > 0 else 0.0

    docs = privacy_metrics["processed_docs"]
    avg_tps = privacy_metrics["sbert_tps_sum"] / docs if docs > 0 else 0
    avg_div = privacy_metrics["lexical_divergence_sum"] / docs if docs > 0 else 0
    avg_rouge1 = privacy_metrics["rouge1_sum"] / docs if docs > 0 else 0
    avg_rougeL = privacy_metrics["rougeL_sum"] / docs if docs > 0 else 0
    avg_llm_no_orig = privacy_metrics["llm_reid_no_orig_sum"] / docs if docs > 0 else 0
    avg_llm_with_orig = privacy_metrics["llm_reid_with_orig_sum"] / docs if docs > 0 else 0

    results_to_save["global"] = {
        "total_tp": total_tp, "total_fp": total_fp, "total_fn": total_fn,
        "precision": round(g_precision, 4),
        "recall": round(g_recall, 4),
        "f1_score": round(g_f1, 4)
    }
    try:
        results_to_save["privacy_and_utility"] = {
            "sbert_tps_avg": round(avg_tps, 4),
            "lexical_divergence_avg": round(avg_div, 4),
            "rouge1_overlap_avg": round(avg_rouge1, 4),
            "rougeL_overlap_avg": round(avg_rougeL, 4),
            f"{MODELO_METRICA}_reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
            f"{MODELO_METRICA}_reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    except:
        results_to_save["privacy_and_utility"] = {
          "sbert_tps_avg": round(avg_tps, 4),
          "lexical_divergence_avg": round(avg_div, 4),
          "rouge1_overlap_avg": round(avg_rouge1, 4),
          "rougeL_overlap_avg": round(avg_rougeL, 4),
          "reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
          "reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    save_dir = f'/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/resultados_v5/{dataset_name}/'
    os.makedirs(save_dir, exist_ok=True)
    saved_in = f'{save_dir}geral_info_{model.save_arq}.json'
    attack_dataset_path = f'{save_dir}attack_dataset_{model.save_arq}.json'
    log_dataset_path = f'{save_dir}log_info_{model.save_arq}.json'

    with open(saved_in, 'w', encoding='utf-8') as f_out:
        json.dump(results_to_save, f_out, indent=4, ensure_ascii=False)

    with open(attack_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(attack_dataset, f_out, indent=4, ensure_ascii=False)

    with open(log_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(log_dataset, f_out, indent=4, ensure_ascii=False)
    try:
        print("=" * 40)
        print(f"F1 Global ({dataset_name}): {g_f1:.4f}")
        print(f"TPS (SBERT): {avg_tps:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Sem Orig): {avg_llm_no_orig:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Com Orig): {avg_llm_with_orig:.4f}")
        print(f"Arquivo de resultados gerado com sucesso: {saved_in}")
        print(f"Arquivo de ataque gerado com sucesso: {attack_dataset_path}")
        print(f"Arquivo de log gerado com sucesso: {log_dataset_path}\n")
    except:
      pass

datasets = [
    #{"name": "first_harem", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/first_harem/first_harem_selective.json"},
    #{"name": "lener_br", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/lener_br/lener_br_converted.json"},
    #{"name": "mariNER", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/mariNER/mariNER_converted.json"},
    #{"name": "mini_harem", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/mini_harem/mini_harem_selective.json"},
    #{"name": "second_harem_selective", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/second_harem/second_harem_selective.json"},
    #{"name": "ulyssesNER_Br_PL_v2_corpus", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/ulyssesNER_Br_PL_v2_corpus/ulyssesCategorias-v2_converted.json"},
]

meu_modelo = NERModel()

for ds in datasets:
    print(f"Iniciando avaliação: {ds['name']}")
    evaluate_model(dataset_path=ds['path'], dataset_name=ds['name'], model=meu_modelo, batch_size=5)

## GLINER 1

In [ ]:
class PrivacyMetricsEvaluator:
    def __init__(self, sbert_device="cuda"):

        self.sbert_model = SentenceTransformer(
            "paraphrase-multilingual-MiniLM-L12-v2",
            device=sbert_device
        )
        self.model_metrica  = carregar_modelo_gpu_4bit(MODELO_METRICA)
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

    def generate_anonymized_text(self, text: str, entities: list) -> str:
        sorted_entities = sorted(
            entities,
            key=lambda x: x["start_offset"],
            reverse=True
        )

        anonymized_text = text

        for ent in sorted_entities:
            start = ent["start_offset"]
            end = ent["end_offset"]

            anonymized_text = (
                anonymized_text[:start]
                + f"[{ent['label']}]"
                + anonymized_text[end:]
            )

        return anonymized_text

    def calculate_sbert_tps(self, original_text: str, anonymized_text: str) -> float:
        with torch.inference_mode():
            emb1 = self.sbert_model.encode(
                original_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            emb2 = self.sbert_model.encode(
                anonymized_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            cosine_scores = util.cos_sim(emb1, emb2)
            resultado = cosine_scores.item()

        del emb1
        del emb2
        del cosine_scores

        return resultado

    def calculate_lexical_divergence(self, original_text: str, anonymized_text: str) -> float:
        set_orig = set(original_text.lower().split())
        set_anon = set(anonymized_text.lower().split())

        intersection = len(set_orig.intersection(set_anon))
        union = len(set_orig.union(set_anon))

        if union == 0:
            return 0.0

        jaccard_similarity = intersection / union

        return 1.0 - jaccard_similarity

    def calculate_rouge_overlap(self, original_text: str, anonymized_text: str) -> dict:
        scores = self.rouge.score(original_text, anonymized_text)

        return {
            "rouge1_fmeasure": scores["rouge1"].fmeasure,
            "rougeL_fmeasure": scores["rougeL"].fmeasure
        }

    def execute_llm_reidentification_attack(
        self,
        original_text: str,
        anonymized_text: str,
        true_entities: list,
        use_original: bool
    ) -> float:
        if not true_entities:
            return 0.0

        original_text_prompt = original_text[:2500]
        anonymized_text_prompt = anonymized_text[:2500]

        if use_original:
            prompt = f"""
Você é um especialista em reidentificação.
Mapeie as tags do texto anonimizado para as palavras reais usando o texto original.

Texto Original:
{original_text_prompt}

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com os valores reais removidos, separados por vírgula.
""".strip()
        else:
            prompt = f"""
Você é um especialista em reidentificação.
Tente adivinhar as palavras originais que foram substituídas por tags no seguinte texto.

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com seus palpites, separados por vírgula.
""".strip()

        mensagens = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        try:
            generated_text = gerar_resposta_chat(
                mensagens=mensagens,
                gerador=self.model_metrica,
                max_new_tokens=MAX_NEW_TOKENS_ATAQUE,
                max_input_tokens=MAX_INPUT_TOKENS_ATAQUE
            ).lower()

        except Exception:
            return 0.0

        success_count = 0

        for ent in true_entities:
            entity_text = original_text[
                ent["start_offset"]:ent["end_offset"]
            ].lower()

            if entity_text in generated_text:
                success_count += 1

        return success_count / len(true_entities)

    def avaliar_texto(self, original_text: str, entities: list) -> dict:
        anonymized_text = self.generate_anonymized_text(
            original_text,
            entities
        )

        sbert_tps = self.calculate_sbert_tps(
            original_text,
            anonymized_text
        )

        lexical_divergence = self.calculate_lexical_divergence(
            original_text,
            anonymized_text
        )

        rouge_scores = self.calculate_rouge_overlap(
            original_text,
            anonymized_text
        )

        ataque_com_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=True
        )

        ataque_sem_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=False
        )

        return {
            "original_text": original_text,
            "anonymized_text": anonymized_text,
            "entities": entities,
            "sbert_tps": sbert_tps,
            "lexical_divergence": lexical_divergence,
            "rouge1_fmeasure": rouge_scores["rouge1_fmeasure"],
            "rougeL_fmeasure": rouge_scores["rougeL_fmeasure"],
            "llm_attack_with_original": ataque_com_original,
            "llm_attack_without_original": ataque_sem_original
        }

class NERModel:
    def __init__(self):
        self.save_arq = 'knowledgator_gliner_x_large'
        self.model_used = GlinerXLarge()
        self.threshold = 0.3
        self.labels = ['person', 'organization', 'local', 'time_or_data', 'value', 'case_law', 'legislation_or_legal_basis', 'product_of_law']
        self.preprocessor = TextPreprocessor(max_chars=1500, overlap_chars=200)

    def predict(self, text: str):
        all_predictions = []
        chunks = self.preprocessor.chunk_with_offsets(text)

        for chunk_data in chunks:

            chunk_text = chunk_data["text"]
            chunk_start = chunk_data["start_offset"]

            pred = self.model_used.extract_entities(chunk_text, self.labels, self.threshold)

            for p in pred:
                adjusted_p = p.copy()
                adjusted_p['start_offset'] += chunk_start
                adjusted_p['end_offset'] += chunk_start

                if adjusted_p['label'] == "person":
                    adjusted_p['label'] = "PESSOA"
                elif adjusted_p['label'] == "organization":
                    adjusted_p['label'] = "ORGANIZACAO"
                elif adjusted_p['label'] == "local":
                    adjusted_p['label'] = "LOCAL"
                elif adjusted_p['label'] == 'time_or_data':
                    adjusted_p['label'] = "TEMPO/DATA"
                elif adjusted_p['label'] == 'value':
                    adjusted_p['label'] = "VALOR"
                elif adjusted_p['label'] == 'case_law':
                    adjusted_p['label'] = "JURISPRUDENCIA"
                elif adjusted_p['label'] == 'legislation_or_legal_basis':
                    adjusted_p['label'] = "LEGISLACAO/FUNDAMENTO"
                elif adjusted_p['label'] == 'product_of_law':
                    adjusted_p['label'] = "PRODUTODELEI"

                is_duplicate = any(
                    adjusted_p['label'] == existing['label'] and
                    abs(adjusted_p['start_offset'] - existing['start_offset']) < 5
                    for existing in all_predictions
                )

                if not is_duplicate:
                    all_predictions.append(adjusted_p)

        return all_predictions

def calculate_metrics(true_entities, pred_entities):
    true_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in true_entities)
    pred_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in pred_entities)

    tp_set = true_set.intersection(pred_set)
    fp_set = pred_set - true_set
    fn_set = true_set - pred_set

    label_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for label, _, _ in tp_set:
        label_metrics[label]['tp'] += 1

    for label, _, _ in fp_set:
        label_metrics[label]['fp'] += 1

    for label, _, _ in fn_set:
        label_metrics[label]['fn'] += 1

    return label_metrics

def evaluate_model(dataset_path, dataset_name, model, batch_size=10):
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    privacy_evaluator = PrivacyMetricsEvaluator()

    global_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    privacy_metrics = {
        "sbert_tps_sum": 0.0,
        "lexical_divergence_sum": 0.0,
        "rouge1_sum": 0.0,
        "rougeL_sum": 0.0,
        "llm_reid_no_orig_sum": 0.0,
        "llm_reid_with_orig_sum": 0.0,
        "processed_docs": 0
    }

    attack_dataset = []
    log_dataset = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Processando {dataset_name}"):
        batch = dataset[i:i + batch_size]

        for doc in batch:
            text = doc.get('doc_text', '')
            true_entities = doc.get('entities', [])
            pred_entities = model.predict(text)

            doc_metrics = calculate_metrics(true_entities, pred_entities)
            for label, counts in doc_metrics.items():
                global_metrics[label]['tp'] += counts['tp']
                global_metrics[label]['fp'] += counts['fp']
                global_metrics[label]['fn'] += counts['fn']

            anonymized_text = privacy_evaluator.generate_anonymized_text(text, pred_entities)

            tps_score = privacy_evaluator.calculate_sbert_tps(text, anonymized_text)
            div_score = privacy_evaluator.calculate_lexical_divergence(text, anonymized_text)
            rouge_scores = privacy_evaluator.calculate_rouge_overlap(text, anonymized_text)

            llm_rate_no_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=False)
            llm_rate_with_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=True)

            privacy_metrics["sbert_tps_sum"] += tps_score
            privacy_metrics["lexical_divergence_sum"] += div_score
            privacy_metrics["rouge1_sum"] += rouge_scores["rouge1_fmeasure"]
            privacy_metrics["rougeL_sum"] += rouge_scores["rougeL_fmeasure"]
            privacy_metrics["llm_reid_no_orig_sum"] += llm_rate_no_orig
            privacy_metrics["llm_reid_with_orig_sum"] += llm_rate_with_orig
            privacy_metrics["processed_docs"] += 1

            attack_dataset.append({
                "original_text": text,
                "anonymized_text": anonymized_text,
                "true_entities": true_entities
            })

            log_dataset.append({
                "original_text": text,
                "true_entities": true_entities,
                "predicted_entities": pred_entities
            })

        gc.collect()

    total_tp = total_fp = total_fn = 0
    results_to_save = {"per_entity": {}, "global": {}, "privacy_and_utility": {}}

    print("=" * 40)
    print(f"RESULTADOS POR ENTIDADE - {dataset_name}")
    print("=" * 40)

    for label, counts in global_metrics.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
        total_tp += tp
        total_fp += fp
        total_fn += fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        print(f"Entidade: [{label}] | F1: {f1:.4f}")

        results_to_save["per_entity"][label] = {
            "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1_score": round(f1, 4)
        }

    g_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    g_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    g_f1 = 2 * (g_precision * g_recall) / (g_precision + g_recall) if (g_precision + g_recall) > 0 else 0.0

    docs = privacy_metrics["processed_docs"]
    avg_tps = privacy_metrics["sbert_tps_sum"] / docs if docs > 0 else 0
    avg_div = privacy_metrics["lexical_divergence_sum"] / docs if docs > 0 else 0
    avg_rouge1 = privacy_metrics["rouge1_sum"] / docs if docs > 0 else 0
    avg_rougeL = privacy_metrics["rougeL_sum"] / docs if docs > 0 else 0
    avg_llm_no_orig = privacy_metrics["llm_reid_no_orig_sum"] / docs if docs > 0 else 0
    avg_llm_with_orig = privacy_metrics["llm_reid_with_orig_sum"] / docs if docs > 0 else 0

    results_to_save["global"] = {
        "total_tp": total_tp, "total_fp": total_fp, "total_fn": total_fn,
        "precision": round(g_precision, 4),
        "recall": round(g_recall, 4),
        "f1_score": round(g_f1, 4)
    }
    try:
        results_to_save["privacy_and_utility"] = {
            "sbert_tps_avg": round(avg_tps, 4),
            "lexical_divergence_avg": round(avg_div, 4),
            "rouge1_overlap_avg": round(avg_rouge1, 4),
            "rougeL_overlap_avg": round(avg_rougeL, 4),
            f"{MODELO_METRICA}_reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
            f"{MODELO_METRICA}_reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    except:
        results_to_save["privacy_and_utility"] = {
          "sbert_tps_avg": round(avg_tps, 4),
          "lexical_divergence_avg": round(avg_div, 4),
          "rouge1_overlap_avg": round(avg_rouge1, 4),
          "rougeL_overlap_avg": round(avg_rougeL, 4),
          "reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
          "reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    save_dir = f'/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/resultados_v5/{dataset_name}/'
    os.makedirs(save_dir, exist_ok=True)
    saved_in = f'{save_dir}geral_info_{model.save_arq}.json'
    attack_dataset_path = f'{save_dir}attack_dataset_{model.save_arq}.json'
    log_dataset_path = f'{save_dir}log_info_{model.save_arq}.json'

    with open(saved_in, 'w', encoding='utf-8') as f_out:
        json.dump(results_to_save, f_out, indent=4, ensure_ascii=False)

    with open(attack_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(attack_dataset, f_out, indent=4, ensure_ascii=False)

    with open(log_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(log_dataset, f_out, indent=4, ensure_ascii=False)
    try:
        print("=" * 40)
        print(f"F1 Global ({dataset_name}): {g_f1:.4f}")
        print(f"TPS (SBERT): {avg_tps:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Sem Orig): {avg_llm_no_orig:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Com Orig): {avg_llm_with_orig:.4f}")
        print(f"Arquivo de resultados gerado com sucesso: {saved_in}")
        print(f"Arquivo de ataque gerado com sucesso: {attack_dataset_path}")
        print(f"Arquivo de log gerado com sucesso: {log_dataset_path}\n")
    except:
      pass

datasets = [
    #{"name": "first_harem", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/first_harem/first_harem_selective.json"},
    #{"name": "lener_br", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/lener_br/lener_br_converted.json"},
    #{"name": "mariNER", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/mariNER/mariNER_converted.json"},
    #{"name": "mini_harem", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/mini_harem/mini_harem_selective.json"},
    #{"name": "second_harem_selective", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/second_harem/second_harem_selective.json"},
    #{"name": "ulyssesNER_Br_PL_v2_corpus", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/ulyssesNER_Br_PL_v2_corpus/ulyssesCategorias-v2_converted.json"},
]

meu_modelo = NERModel()

for ds in datasets:
    print(f"Iniciando avaliação: {ds['name']}")
    evaluate_model(dataset_path=ds['path'], dataset_name=ds['name'], model=meu_modelo, batch_size=5)

## BERT MODELS

In [ ]:
class PrivacyMetricsEvaluator:
    def __init__(self, sbert_device="cuda"):

        self.sbert_model = SentenceTransformer(
            "paraphrase-multilingual-MiniLM-L12-v2",
            device=sbert_device
        )
        self.model_metrica  = carregar_modelo_gpu_4bit(MODELO_METRICA)
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

    def generate_anonymized_text(self, text: str, entities: list) -> str:
        sorted_entities = sorted(
            entities,
            key=lambda x: x["start_offset"],
            reverse=True
        )

        anonymized_text = text

        for ent in sorted_entities:
            start = ent["start_offset"]
            end = ent["end_offset"]

            anonymized_text = (
                anonymized_text[:start]
                + f"[{ent['label']}]"
                + anonymized_text[end:]
            )

        return anonymized_text

    def calculate_sbert_tps(self, original_text: str, anonymized_text: str) -> float:
        with torch.inference_mode():
            emb1 = self.sbert_model.encode(
                original_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            emb2 = self.sbert_model.encode(
                anonymized_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            cosine_scores = util.cos_sim(emb1, emb2)
            resultado = cosine_scores.item()

        del emb1
        del emb2
        del cosine_scores

        return resultado

    def calculate_lexical_divergence(self, original_text: str, anonymized_text: str) -> float:
        set_orig = set(original_text.lower().split())
        set_anon = set(anonymized_text.lower().split())

        intersection = len(set_orig.intersection(set_anon))
        union = len(set_orig.union(set_anon))

        if union == 0:
            return 0.0

        jaccard_similarity = intersection / union

        return 1.0 - jaccard_similarity

    def calculate_rouge_overlap(self, original_text: str, anonymized_text: str) -> dict:
        scores = self.rouge.score(original_text, anonymized_text)

        return {
            "rouge1_fmeasure": scores["rouge1"].fmeasure,
            "rougeL_fmeasure": scores["rougeL"].fmeasure
        }

    def execute_llm_reidentification_attack(
        self,
        original_text: str,
        anonymized_text: str,
        true_entities: list,
        use_original: bool
    ) -> float:
        if not true_entities:
            return 0.0

        original_text_prompt = original_text[:2500]
        anonymized_text_prompt = anonymized_text[:2500]

        if use_original:
            prompt = f"""
Você é um especialista em reidentificação.
Mapeie as tags do texto anonimizado para as palavras reais usando o texto original.

Texto Original:
{original_text_prompt}

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com os valores reais removidos, separados por vírgula.
""".strip()
        else:
            prompt = f"""
Você é um especialista em reidentificação.
Tente adivinhar as palavras originais que foram substituídas por tags no seguinte texto.

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com seus palpites, separados por vírgula.
""".strip()

        mensagens = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        try:
            generated_text = gerar_resposta_chat(
                mensagens=mensagens,
                gerador=self.model_metrica,
                max_new_tokens=MAX_NEW_TOKENS_ATAQUE,
                max_input_tokens=MAX_INPUT_TOKENS_ATAQUE
            ).lower()

        except Exception:
            return 0.0

        success_count = 0

        for ent in true_entities:
            entity_text = original_text[
                ent["start_offset"]:ent["end_offset"]
            ].lower()

            if entity_text in generated_text:
                success_count += 1

        return success_count / len(true_entities)

    def avaliar_texto(self, original_text: str, entities: list) -> dict:
        anonymized_text = self.generate_anonymized_text(
            original_text,
            entities
        )

        sbert_tps = self.calculate_sbert_tps(
            original_text,
            anonymized_text
        )

        lexical_divergence = self.calculate_lexical_divergence(
            original_text,
            anonymized_text
        )

        rouge_scores = self.calculate_rouge_overlap(
            original_text,
            anonymized_text
        )

        ataque_com_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=True
        )

        ataque_sem_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=False
        )

        return {
            "original_text": original_text,
            "anonymized_text": anonymized_text,
            "entities": entities,
            "sbert_tps": sbert_tps,
            "lexical_divergence": lexical_divergence,
            "rouge1_fmeasure": rouge_scores["rouge1_fmeasure"],
            "rougeL_fmeasure": rouge_scores["rougeL_fmeasure"],
            "llm_attack_with_original": ataque_com_original,
            "llm_attack_without_original": ataque_sem_original
        }

class NERModel:
    def __init__(self):
        self.save_arq = 'Babelscape_wikineural_multilingual_ner'
        self.model_used = wikineural_multi_ner()
        self.preprocessor = TextPreprocessor(max_chars=1500, overlap_chars=200)

    def predict(self, text: str):
        all_predictions = []
        chunks = self.preprocessor.chunk_with_offsets(text)

        for chunk_data in chunks:
            chunk_text = chunk_data["text"]
            chunk_start = chunk_data["start_offset"]

            pred = self.model_used.extract_entities(chunk_text)

            for p in pred:

                adjusted_p = p.copy()
                adjusted_p['start_offset'] += chunk_start
                adjusted_p['end_offset'] += chunk_start

                if adjusted_p['label'] == "PER":
                    adjusted_p['label'] = "PESSOA"
                elif adjusted_p['label'] == "ORG":
                    adjusted_p['label'] = "ORGANIZACAO"
                elif adjusted_p['label'] == "LOC":
                    adjusted_p['label'] = "LOCAL"

                is_duplicate = any(
                    adjusted_p['label'] == existing['label'] and
                    abs(adjusted_p['start_offset'] - existing['start_offset']) < 5
                    for existing in all_predictions
                )

                if not is_duplicate:
                    all_predictions.append(adjusted_p)

        return all_predictions
def calculate_metrics(true_entities, pred_entities):
    true_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in true_entities)
    pred_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in pred_entities)

    tp_set = true_set.intersection(pred_set)
    fp_set = pred_set - true_set
    fn_set = true_set - pred_set

    label_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for label, _, _ in tp_set:
        label_metrics[label]['tp'] += 1

    for label, _, _ in fp_set:
        label_metrics[label]['fp'] += 1

    for label, _, _ in fn_set:
        label_metrics[label]['fn'] += 1

    return label_metrics

def evaluate_model(dataset_path, dataset_name, model, batch_size=10):
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    privacy_evaluator = PrivacyMetricsEvaluator()

    global_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    privacy_metrics = {
        "sbert_tps_sum": 0.0,
        "lexical_divergence_sum": 0.0,
        "rouge1_sum": 0.0,
        "rougeL_sum": 0.0,
        "llm_reid_no_orig_sum": 0.0,
        "llm_reid_with_orig_sum": 0.0,
        "processed_docs": 0
    }

    attack_dataset = []
    log_dataset = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Processando {dataset_name}"):
        batch = dataset[i:i + batch_size]

        for doc in batch:
            text = doc.get('doc_text', '')
            true_entities = doc.get('entities', [])
            pred_entities = model.predict(text)

            doc_metrics = calculate_metrics(true_entities, pred_entities)
            for label, counts in doc_metrics.items():
                global_metrics[label]['tp'] += counts['tp']
                global_metrics[label]['fp'] += counts['fp']
                global_metrics[label]['fn'] += counts['fn']

            anonymized_text = privacy_evaluator.generate_anonymized_text(text, pred_entities)

            tps_score = privacy_evaluator.calculate_sbert_tps(text, anonymized_text)
            div_score = privacy_evaluator.calculate_lexical_divergence(text, anonymized_text)
            rouge_scores = privacy_evaluator.calculate_rouge_overlap(text, anonymized_text)

            llm_rate_no_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=False)
            llm_rate_with_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=True)

            privacy_metrics["sbert_tps_sum"] += tps_score
            privacy_metrics["lexical_divergence_sum"] += div_score
            privacy_metrics["rouge1_sum"] += rouge_scores["rouge1_fmeasure"]
            privacy_metrics["rougeL_sum"] += rouge_scores["rougeL_fmeasure"]
            privacy_metrics["llm_reid_no_orig_sum"] += llm_rate_no_orig
            privacy_metrics["llm_reid_with_orig_sum"] += llm_rate_with_orig
            privacy_metrics["processed_docs"] += 1

            attack_dataset.append({
                "original_text": text,
                "anonymized_text": anonymized_text,
                "true_entities": true_entities
            })

            log_dataset.append({
                "original_text": text,
                "true_entities": true_entities,
                "predicted_entities": pred_entities
            })

        gc.collect()

    total_tp = total_fp = total_fn = 0
    results_to_save = {"per_entity": {}, "global": {}, "privacy_and_utility": {}}

    print("=" * 40)
    print(f"RESULTADOS POR ENTIDADE - {dataset_name}")
    print("=" * 40)

    for label, counts in global_metrics.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
        total_tp += tp
        total_fp += fp
        total_fn += fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        print(f"Entidade: [{label}] | F1: {f1:.4f}")

        results_to_save["per_entity"][label] = {
            "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1_score": round(f1, 4)
        }

    g_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    g_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    g_f1 = 2 * (g_precision * g_recall) / (g_precision + g_recall) if (g_precision + g_recall) > 0 else 0.0

    docs = privacy_metrics["processed_docs"]
    avg_tps = privacy_metrics["sbert_tps_sum"] / docs if docs > 0 else 0
    avg_div = privacy_metrics["lexical_divergence_sum"] / docs if docs > 0 else 0
    avg_rouge1 = privacy_metrics["rouge1_sum"] / docs if docs > 0 else 0
    avg_rougeL = privacy_metrics["rougeL_sum"] / docs if docs > 0 else 0
    avg_llm_no_orig = privacy_metrics["llm_reid_no_orig_sum"] / docs if docs > 0 else 0
    avg_llm_with_orig = privacy_metrics["llm_reid_with_orig_sum"] / docs if docs > 0 else 0

    results_to_save["global"] = {
        "total_tp": total_tp, "total_fp": total_fp, "total_fn": total_fn,
        "precision": round(g_precision, 4),
        "recall": round(g_recall, 4),
        "f1_score": round(g_f1, 4)
    }
    try:
        results_to_save["privacy_and_utility"] = {
            "sbert_tps_avg": round(avg_tps, 4),
            "lexical_divergence_avg": round(avg_div, 4),
            "rouge1_overlap_avg": round(avg_rouge1, 4),
            "rougeL_overlap_avg": round(avg_rougeL, 4),
            f"{MODELO_METRICA}_reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
            f"{MODELO_METRICA}_reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    except:
        results_to_save["privacy_and_utility"] = {
          "sbert_tps_avg": round(avg_tps, 4),
          "lexical_divergence_avg": round(avg_div, 4),
          "rouge1_overlap_avg": round(avg_rouge1, 4),
          "rougeL_overlap_avg": round(avg_rougeL, 4),
          "reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
          "reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    save_dir = f'/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/resultados_v5/{dataset_name}/'
    os.makedirs(save_dir, exist_ok=True)
    saved_in = f'{save_dir}geral_info_{model.save_arq}.json'
    attack_dataset_path = f'{save_dir}attack_dataset_{model.save_arq}.json'
    log_dataset_path = f'{save_dir}log_info_{model.save_arq}.json'

    with open(saved_in, 'w', encoding='utf-8') as f_out:
        json.dump(results_to_save, f_out, indent=4, ensure_ascii=False)

    with open(attack_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(attack_dataset, f_out, indent=4, ensure_ascii=False)

    with open(log_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(log_dataset, f_out, indent=4, ensure_ascii=False)
    try:
        print("=" * 40)
        print(f"F1 Global ({dataset_name}): {g_f1:.4f}")
        print(f"TPS (SBERT): {avg_tps:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Sem Orig): {avg_llm_no_orig:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Com Orig): {avg_llm_with_orig:.4f}")
        print(f"Arquivo de resultados gerado com sucesso: {saved_in}")
        print(f"Arquivo de ataque gerado com sucesso: {attack_dataset_path}")
        print(f"Arquivo de log gerado com sucesso: {log_dataset_path}\n")
    except:
      pass

datasets = [
    #{"name": "first_harem", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/first_harem/first_harem_selective.json"},
    #{"name": "lener_br", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/lener_br/lener_br_converted.json"},
    #{"name": "mariNER", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/mariNER/mariNER_converted.json"},
    #{"name": "mini_harem", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/mini_harem/mini_harem_selective.json"},
    #{"name": "second_harem_selective", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/second_harem/second_harem_selective.json"},
    #{"name": "ulyssesNER_Br_PL_v2_corpus", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/ulyssesNER_Br_PL_v2_corpus/ulyssesCategorias-v2_converted.json"},
]

meu_modelo = NERModel()

for ds in datasets:
    print(f"Iniciando avaliação: {ds['name']}")
    evaluate_model(dataset_path=ds['path'], dataset_name=ds['name'], model=meu_modelo, batch_size=5)

## SLMs

In [ ]:
class PrivacyMetricsEvaluator:
    def __init__(self, sbert_device="cuda"):

        self.sbert_model = SentenceTransformer(
            "paraphrase-multilingual-MiniLM-L12-v2",
            device=sbert_device
        )
        self.model_metrica  = carregar_modelo_gpu_4bit(MODELO_METRICA)
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

    def generate_anonymized_text(self, text: str, entities: list) -> str:
        sorted_entities = sorted(
            entities,
            key=lambda x: x["start_offset"],
            reverse=True
        )

        anonymized_text = text

        for ent in sorted_entities:
            start = ent["start_offset"]
            end = ent["end_offset"]

            anonymized_text = (
                anonymized_text[:start]
                + f"[{ent['label']}]"
                + anonymized_text[end:]
            )

        return anonymized_text

    def calculate_sbert_tps(self, original_text: str, anonymized_text: str) -> float:
        with torch.inference_mode():
            emb1 = self.sbert_model.encode(
                original_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            emb2 = self.sbert_model.encode(
                anonymized_text,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

            cosine_scores = util.cos_sim(emb1, emb2)
            resultado = cosine_scores.item()

        del emb1
        del emb2
        del cosine_scores

        return resultado

    def calculate_lexical_divergence(self, original_text: str, anonymized_text: str) -> float:
        set_orig = set(original_text.lower().split())
        set_anon = set(anonymized_text.lower().split())

        intersection = len(set_orig.intersection(set_anon))
        union = len(set_orig.union(set_anon))

        if union == 0:
            return 0.0

        jaccard_similarity = intersection / union

        return 1.0 - jaccard_similarity

    def calculate_rouge_overlap(self, original_text: str, anonymized_text: str) -> dict:
        scores = self.rouge.score(original_text, anonymized_text)

        return {
            "rouge1_fmeasure": scores["rouge1"].fmeasure,
            "rougeL_fmeasure": scores["rougeL"].fmeasure
        }

    def execute_llm_reidentification_attack(
        self,
        original_text: str,
        anonymized_text: str,
        true_entities: list,
        use_original: bool
    ) -> float:
        if not true_entities:
            return 0.0

        original_text_prompt = original_text[:2500]
        anonymized_text_prompt = anonymized_text[:2500]

        if use_original:
            prompt = f"""
Você é um especialista em reidentificação.
Mapeie as tags do texto anonimizado para as palavras reais usando o texto original.

Texto Original:
{original_text_prompt}

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com os valores reais removidos, separados por vírgula.
""".strip()
        else:
            prompt = f"""
Você é um especialista em reidentificação.
Tente adivinhar as palavras originais que foram substituídas por tags no seguinte texto.

Texto Anonimizado:
{anonymized_text_prompt}

Responda apenas com seus palpites, separados por vírgula.
""".strip()

        mensagens = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        try:
            generated_text = gerar_resposta_chat(
                mensagens=mensagens,
                gerador=self.model_metrica,
                max_new_tokens=MAX_NEW_TOKENS_ATAQUE,
                max_input_tokens=MAX_INPUT_TOKENS_ATAQUE
            ).lower()

        except Exception:
            return 0.0

        success_count = 0

        for ent in true_entities:
            entity_text = original_text[
                ent["start_offset"]:ent["end_offset"]
            ].lower()

            if entity_text in generated_text:
                success_count += 1

        return success_count / len(true_entities)

    def avaliar_texto(self, original_text: str, entities: list) -> dict:
        anonymized_text = self.generate_anonymized_text(
            original_text,
            entities
        )

        sbert_tps = self.calculate_sbert_tps(
            original_text,
            anonymized_text
        )

        lexical_divergence = self.calculate_lexical_divergence(
            original_text,
            anonymized_text
        )

        rouge_scores = self.calculate_rouge_overlap(
            original_text,
            anonymized_text
        )

        ataque_com_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=True
        )

        ataque_sem_original = self.execute_llm_reidentification_attack(
            original_text=original_text,
            anonymized_text=anonymized_text,
            true_entities=entities,
            use_original=False
        )

        return {
            "original_text": original_text,
            "anonymized_text": anonymized_text,
            "entities": entities,
            "sbert_tps": sbert_tps,
            "lexical_divergence": lexical_divergence,
            "rouge1_fmeasure": rouge_scores["rouge1_fmeasure"],
            "rougeL_fmeasure": rouge_scores["rougeL_fmeasure"],
            "llm_attack_with_original": ataque_com_original,
            "llm_attack_without_original": ataque_sem_original
        }

class NERModel:
    def __init__(self):
        self.save_arq = 'meta-llama_Llama-3.1-8B-Instruct_4bit'
        self.model_used = carregar_modelo_gpu_4bit(MODELO_NER)


    def predict(self, text: str):
        pred = extrair_entidades(text, self.model_used)
        print(f"\n{pred}")
        return pred



def calculate_metrics(true_entities, pred_entities):
    true_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in true_entities)
    pred_set = set((ent['label'], ent['start_offset'], ent['end_offset']) for ent in pred_entities)

    tp_set = true_set.intersection(pred_set)
    fp_set = pred_set - true_set
    fn_set = true_set - pred_set

    label_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for label, _, _ in tp_set:
        label_metrics[label]['tp'] += 1

    for label, _, _ in fp_set:
        label_metrics[label]['fp'] += 1

    for label, _, _ in fn_set:
        label_metrics[label]['fn'] += 1

    return label_metrics

def evaluate_model(dataset_path, dataset_name, model, batch_size=10):
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    privacy_evaluator = PrivacyMetricsEvaluator()

    global_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    privacy_metrics = {
        "sbert_tps_sum": 0.0,
        "lexical_divergence_sum": 0.0,
        "rouge1_sum": 0.0,
        "rougeL_sum": 0.0,
        "llm_reid_no_orig_sum": 0.0,
        "llm_reid_with_orig_sum": 0.0,
        "processed_docs": 0
    }

    attack_dataset = []
    log_dataset = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Processando {dataset_name}"):
        batch = dataset[i:i + batch_size]

        for doc in batch:
            text = doc.get('doc_text', '')
            true_entities = doc.get('entities', [])
            pred_entities = model.predict(text)

            doc_metrics = calculate_metrics(true_entities, pred_entities)
            for label, counts in doc_metrics.items():
                global_metrics[label]['tp'] += counts['tp']
                global_metrics[label]['fp'] += counts['fp']
                global_metrics[label]['fn'] += counts['fn']

            anonymized_text = privacy_evaluator.generate_anonymized_text(text, pred_entities)

            tps_score = privacy_evaluator.calculate_sbert_tps(text, anonymized_text)
            div_score = privacy_evaluator.calculate_lexical_divergence(text, anonymized_text)
            rouge_scores = privacy_evaluator.calculate_rouge_overlap(text, anonymized_text)

            llm_rate_no_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=False)
            llm_rate_with_orig = privacy_evaluator.execute_llm_reidentification_attack(text, anonymized_text, true_entities, use_original=True)

            privacy_metrics["sbert_tps_sum"] += tps_score
            privacy_metrics["lexical_divergence_sum"] += div_score
            privacy_metrics["rouge1_sum"] += rouge_scores["rouge1_fmeasure"]
            privacy_metrics["rougeL_sum"] += rouge_scores["rougeL_fmeasure"]
            privacy_metrics["llm_reid_no_orig_sum"] += llm_rate_no_orig
            privacy_metrics["llm_reid_with_orig_sum"] += llm_rate_with_orig
            privacy_metrics["processed_docs"] += 1

            attack_dataset.append({
                "original_text": text,
                "anonymized_text": anonymized_text,
                "true_entities": true_entities
            })

            log_dataset.append({
                "original_text": text,
                "true_entities": true_entities,
                "predicted_entities": pred_entities
            })

        gc.collect()

    total_tp = total_fp = total_fn = 0
    results_to_save = {"per_entity": {}, "global": {}, "privacy_and_utility": {}}

    print("=" * 40)
    print(f"RESULTADOS POR ENTIDADE - {dataset_name}")
    print("=" * 40)

    for label, counts in global_metrics.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
        total_tp += tp
        total_fp += fp
        total_fn += fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        print(f"Entidade: [{label}] | F1: {f1:.4f}")

        results_to_save["per_entity"][label] = {
            "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1_score": round(f1, 4)
        }

    g_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    g_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    g_f1 = 2 * (g_precision * g_recall) / (g_precision + g_recall) if (g_precision + g_recall) > 0 else 0.0

    docs = privacy_metrics["processed_docs"]
    avg_tps = privacy_metrics["sbert_tps_sum"] / docs if docs > 0 else 0
    avg_div = privacy_metrics["lexical_divergence_sum"] / docs if docs > 0 else 0
    avg_rouge1 = privacy_metrics["rouge1_sum"] / docs if docs > 0 else 0
    avg_rougeL = privacy_metrics["rougeL_sum"] / docs if docs > 0 else 0
    avg_llm_no_orig = privacy_metrics["llm_reid_no_orig_sum"] / docs if docs > 0 else 0
    avg_llm_with_orig = privacy_metrics["llm_reid_with_orig_sum"] / docs if docs > 0 else 0

    results_to_save["global"] = {
        "total_tp": total_tp, "total_fp": total_fp, "total_fn": total_fn,
        "precision": round(g_precision, 4),
        "recall": round(g_recall, 4),
        "f1_score": round(g_f1, 4)
    }
    try:
        results_to_save["privacy_and_utility"] = {
            "sbert_tps_avg": round(avg_tps, 4),
            "lexical_divergence_avg": round(avg_div, 4),
            "rouge1_overlap_avg": round(avg_rouge1, 4),
            "rougeL_overlap_avg": round(avg_rougeL, 4),
            f"{MODELO_METRICA}_reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
            f"{MODELO_METRICA}_reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    except:
        results_to_save["privacy_and_utility"] = {
          "sbert_tps_avg": round(avg_tps, 4),
          "lexical_divergence_avg": round(avg_div, 4),
          "rouge1_overlap_avg": round(avg_rouge1, 4),
          "rougeL_overlap_avg": round(avg_rougeL, 4),
          "reid_success_rate_no_orig": round(avg_llm_no_orig, 4),
          "reid_success_rate_with_orig": round(avg_llm_with_orig, 4)
    }
    save_dir = f'/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/resultados_v5/{dataset_name}/'
    os.makedirs(save_dir, exist_ok=True)
    saved_in = f'{save_dir}geral_info_{model.save_arq}.json'
    attack_dataset_path = f'{save_dir}attack_dataset_{model.save_arq}.json'
    log_dataset_path = f'{save_dir}log_info_{model.save_arq}.json'

    with open(saved_in, 'w', encoding='utf-8') as f_out:
        json.dump(results_to_save, f_out, indent=4, ensure_ascii=False)

    with open(attack_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(attack_dataset, f_out, indent=4, ensure_ascii=False)

    with open(log_dataset_path, 'w', encoding='utf-8') as f_out:
        json.dump(log_dataset, f_out, indent=4, ensure_ascii=False)
    try:
        print("=" * 40)
        print(f"F1 Global ({dataset_name}): {g_f1:.4f}")
        print(f"TPS (SBERT): {avg_tps:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Sem Orig): {avg_llm_no_orig:.4f}")
        print(f"{MODELO_METRICA} Reid Rate (Com Orig): {avg_llm_with_orig:.4f}")
        print(f"Arquivo de resultados gerado com sucesso: {saved_in}")
        print(f"Arquivo de ataque gerado com sucesso: {attack_dataset_path}")
        print(f"Arquivo de log gerado com sucesso: {log_dataset_path}\n")
    except:
      pass

datasets = [
    #{"name": "first_harem", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/first_harem/first_harem_selective.json"},
    #{"name": "lener_br", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/lener_br/lener_br_converted.json"},
    #{"name": "mariNER", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/mariNER/mariNER_converted.json"},
    #{"name": "mini_harem", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/mini_harem/mini_harem_selective.json"},
    #{"name": "second_harem_selective", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/second_harem/second_harem_selective.json"},
    #{"name": "ulyssesNER_Br_PL_v2_corpus", "path": "/content/drive/MyDrive/ANONIMIZAÇÃO E TCC/CODIGOS FINAIS/datasets/json_format_v2/ulyssesNER_Br_PL_v2_corpus/ulyssesCategorias-v2_converted.json"},
]

meu_modelo = NERModel()

for ds in datasets:
    print(f"Iniciando avaliação: {ds['name']}")
    evaluate_model(dataset_path=ds['path'], dataset_name=ds['name'], model=meu_modelo, batch_size=5)